# 02 · Preparación de los datos

**Proyecto:** videos en tendencia de YouTube en Estados Unidos
**Metodología:** CRISP-DM, etapa 3 (preparación de los datos)

Este notebook aplica las ocho decisiones de limpieza del notebook 01 y produce los dos datasets que usan los notebooks siguientes:

- `data/processed/daily.parquet`: una fila por video y día, ya limpia.
- `data/processed/videos.parquet`: **una fila por video**, con las variables para el análisis y los modelos.

La lógica vive en `src/prep.py` para poder reutilizarla. Aquí se ejecuta paso a paso, mostrando el efecto de cada decisión.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src import prep
from src.data import PROCESSED_DIR, load_raw

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.3f}".format)

raw = load_raw()
log = [("Datos originales", len(raw), raw["video_id"].nunique())]
print(f"{len(raw):,} filas · {raw['video_id'].nunique():,} videos")

40,949 filas · 6,351 videos


## 1. Limpieza a nivel de fila (video y día)

### 1.1 Duplicados

Se eliminan las filas idénticas. Cuando un video aparece dos veces el mismo día con cifras distintas, se conserva la de más vistas: como las métricas son acumuladas, es la medición más reciente.

In [2]:
daily = prep.drop_duplicates(raw)
log.append(("Sin duplicados", len(daily), daily["video_id"].nunique()))

print(f"Filas eliminadas: {len(raw) - len(daily)}")
assert not daily.duplicated(["video_id", "trending_date"]).any()

Filas eliminadas: 50


### 1.2 Videos retirados o con error

Un video retirado no tiene métricas confiables ni representa contenido que la empresa quiera imitar. Se excluye el video completo, no solo los días marcados.

In [3]:
before = daily
daily = prep.drop_removed_videos(daily)
log.append(("Sin videos retirados", len(daily), daily["video_id"].nunique()))

print(f"Videos excluidos: {before['video_id'].nunique() - daily['video_id'].nunique()} "
      f"({len(before) - len(daily)} filas)")

Videos excluidos: 4 (26 filas)


### 1.3 Texto vacío

Las etiquetas vacías vienen como el texto `[none]` y las descripciones vacías como nulo. Ambas se llevan a texto vacío para que contar etiquetas o medir longitudes dé cero y no un valor engañoso.

In [4]:
daily = prep.clean_text(daily)
print("Filas sin etiquetas:", (daily["tags"] == "").sum())
print("Filas sin descripción:", (daily["description"] == "").sum())
print("Nulos restantes en texto:", daily[["title", "tags", "description"]].isna().sum().sum())

Filas sin etiquetas: 1518
Filas sin descripción: 567
Nulos restantes en texto: 0


### 1.4 Métricas desactivadas

Cuando un canal desactiva las valoraciones o los comentarios, el dataset registra cero. Ese cero significa "no se pudo medir". Se reemplaza por faltante para no bajar artificialmente las tasas de interacción de esos videos.

In [5]:
daily = prep.mask_disabled_metrics(daily)

pd.DataFrame(
    {
        "faltantes": daily[prep.METRICS].isna().sum(),
        "ceros reales": (daily[prep.METRICS] == 0).sum(),
    }
)

,faltantes,ceros reales
views,0,0
likes,169,3
dislikes,169,214
comment_count,632,124


Los ceros que quedan son reales: videos con valoraciones o comentarios activos que no recibieron ninguno.

## 2. De filas diarias a una fila por video

Para cada video se guarda:

- **Atributos** (título, canal, categoría, etiquetas, descripción): los del **primer día** en tendencia, que son los que tenía cuando llegó.
- **Métricas**: las del primer día (`_first`, cómo llegó) y las del último (`_last`, cuánto alcanzó).
- **Duración**: días distintos en tendencia, primera y última fecha.
- **Velocidad**: días entre la publicación y la primera aparición en tendencia.

In [6]:
videos = prep.build_video_table(daily)

print(f"{len(videos):,} videos · {videos.shape[1]} columnas")
assert videos["video_id"].is_unique
assert len(videos) == daily["video_id"].nunique()
videos[["video_id", "title", "category", "days_trending", "days_to_trending",
        "views_first", "views_last"]].head()

6,347 videos · 24 columnas


,video_id,title,category,days_trending,days_to_trending,views_first,views_last
0,-0CMnp02rNY,Mindy Kaling's Daughter Had the Perfect Reacti...,Entertainment,6,2,475965,800359
1,-0NYY8cqdiQ,Megan Mullally Didn't Notice the Interesting P...,Entertainment,1,3,563746,563746
2,-1Hm41N0dUs,Cast of Avengers: Infinity War Draws Their Cha...,Comedy,3,2,1566807,2058516
3,-1yT-K3c6YI,YOUTUBER QUIZ + TRUTH OR DARE W/ THE MERRELL T...,People & Blogs,4,1,129360,231341
4,-2RVw2_QyxQ,2017 Champions Showdown: Day 3,Education,3,2,67429,71089


### 2.1 Videos antiguos que resurgen

Se marcan con `is_old_video` los videos que tardaron más de 30 días en llegar a tendencia. No se eliminan: son reales, pero su fecha de publicación no dice nada sobre cuándo conviene publicar, así que se excluirán del análisis de fecha.

In [7]:
print(f"Videos antiguos: {videos['is_old_video'].sum()} ({videos['is_old_video'].mean():.1%})")
videos.loc[videos["is_old_video"], "days_to_trending"].describe()[["min", "50%", "max"]].to_frame().T

Videos antiguos: 101 (1.6%)


,min,50%,max
days_to_trending,32.000,"1,116.000","4,215.000"


### 2.2 Videos con duración incompleta

La duración en tendencia no se observa completa en tres casos:

- el video ya estaba en tendencia el **primer día** del dataset (no se sabe cuánto llevaba);
- seguía en tendencia el **último día** (no se sabe cuánto más duró);
- estaba en tendencia junto a uno de los **ocho días sin datos**.

Se marcan con `duration_incomplete`. Sirven para analizar vistas y likes, pero se excluirán cuando la variable de interés sea la duración.

In [8]:
gaps = prep.missing_days(daily)
print("Días sin datos:", ", ".join(gaps.strftime("%d-%m-%Y")))
print(f"Videos con duración incompleta: {videos['duration_incomplete'].sum()} "
      f"({videos['duration_incomplete'].mean():.1%})")

Días sin datos: 10-01-2018, 11-01-2018, 08-04-2018, 09-04-2018, 10-04-2018, 11-04-2018, 12-04-2018, 13-04-2018
Videos con duración incompleta: 1046 (16.5%)


## 3. Construcción de variables

### 3.1 Momento de publicación

`publish_time` viene en UTC. Se convierte a la hora del este de Estados Unidos (Nueva York), la zona horaria con más población del país y la referencia habitual de los medios. Sin esta conversión, un video publicado un lunes a las 9 p. m. en Nueva York aparecería como publicado el martes.

In [9]:
videos = prep.add_publish_features(videos)

comparison = pd.DataFrame(
    {
        "UTC": videos["publish_time"].dt.dayofweek.map(dict(enumerate(prep.DAY_NAMES))).value_counts(),
        "Hora del este": videos["publish_day_name"].value_counts(),
    }
).reindex(prep.DAY_NAMES)
comparison["diferencia"] = comparison["Hora del este"] - comparison["UTC"]
comparison

,UTC,Hora del este,diferencia
Lunes,985,945,-40
Martes,1061,1090,29
Miércoles,1096,1069,-27
Jueves,1054,1042,-12
Viernes,1047,1016,-31
Sábado,546,522,-24
Domingo,558,663,105


### 3.2 Título, etiquetas y descripción

Son las variables que el equipo de contenido puede controlar al publicar: longitud del título, proporción de mayúsculas, signos de exclamación o pregunta, números, cantidad de etiquetas, longitud de la descripción y enlaces que contiene.

In [10]:
videos = prep.add_text_features(videos)

text_cols = ["title_length", "title_words", "title_upper_ratio", "n_tags",
             "description_length", "description_links"]
display(videos[text_cols].describe().T[["min", "50%", "mean", "max"]])

flags = ["title_has_exclamation", "title_has_question", "title_has_digit"]
videos[flags].mean().mul(100).rename("% de videos").to_frame()

,min,50%,mean,max
title_length,3.000,48.000,49.777,100.000
title_words,1.000,8.000,8.711,23.000
title_upper_ratio,0.000,0.195,0.265,1.000
n_tags,0.000,19.000,19.818,69.000
description_length,0.000,805.000,996.784,"5,123.000"
description_links,0.000,6.000,7.573,82.000


,% de videos
title_has_exclamation,10.651
title_has_question,5.656
title_has_digit,26.154


### 3.3 Interacción

Las vistas miden alcance; no dicen si el video generó respuesta. Se agregan tasas relativas a las vistas, que permiten comparar videos de tamaños muy distintos:

- `like_rate`: likes por vista.
- `comment_rate`: comentarios por vista.
- `dislike_share`: proporción de valoraciones negativas.
- `views_growth`: cuánto se multiplicaron las vistas entre el primer y el último día.

Las métricas se guardan también en logaritmo (`log_...`) por la fuerte asimetría vista en el notebook 01.

In [11]:
videos = prep.add_engagement_features(videos)
videos[["like_rate", "comment_rate", "dislike_share", "views_growth"]].describe().T[["min", "50%", "mean", "max"]]

,min,50%,mean,max
like_rate,0.000,0.024,0.030,0.220
comment_rate,0.000,0.003,0.004,0.094
dislike_share,0.000,0.035,0.076,1.000
views_growth,0.547,1.628,2.833,204.047


### 3.4 Historial del canal

`channel_videos` cuenta cuántos videos del mismo canal llegaron a tendencia en el periodo. Es útil para describir y segmentar. En el modelo predictivo se recalculará solo con los datos de entrenamiento, porque calculada sobre todo el periodo usa información del futuro.

In [12]:
videos = prep.add_channel_features(videos)

channels = videos.groupby("channel_title")["video_id"].count()
print(f"Canales: {len(channels):,}")
print(f"Canales con un solo video en tendencia: {(channels == 1).mean():.0%}")
print(f"El 10% de los canales con más videos aporta el "
      f"{channels.nlargest(int(len(channels) * 0.1)).sum() / len(videos):.0%} de los videos en tendencia")

Canales: 2,196
Canales con un solo video en tendencia: 63%
El 10% de los canales con más videos aporta el 51% de los videos en tendencia


## 4. Validación del resultado

In [13]:
log.append(("Dataset por video", len(videos), videos["video_id"].nunique()))
pd.DataFrame(log, columns=["paso", "filas", "videos"]).set_index("paso")

,filas,videos
paso,,
Datos originales,40949,6351
Sin duplicados,40899,6351
Sin videos retirados,40873,6347
Dataset por video,6347,6347


In [14]:
assert videos["video_id"].is_unique
assert (videos["views_last"] > 0).all()
assert (videos["days_trending"] >= 1).all()
assert (videos["days_to_trending"] >= 0).all()
assert videos["category"].notna().all()
assert videos["like_rate"].dropna().between(0, 1).all()
assert videos["dislike_share"].dropna().between(0, 1).all()

nulls = videos.isna().sum()
print("Todas las validaciones pasaron.")
print("\nColumnas con faltantes (canales que desactivaron valoraciones o comentarios):")
nulls[nulls > 0].to_frame("faltantes")

Todas las validaciones pasaron.

Columnas con faltantes (canales que desactivaron valoraciones o comentarios):


,faltantes
likes_first,24
dislikes_first,24
comment_count_first,100
likes_last,24
dislikes_last,24
comment_count_last,100
like_rate,24
comment_rate,100
dislike_share,25
log_likes_first,24


Los únicos faltantes son los esperados: 24 videos sin valoraciones y 100 sin comentarios.

## 5. Guardado

In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
daily.to_parquet(PROCESSED_DIR / "daily.parquet", index=False)
videos.to_parquet(PROCESSED_DIR / "videos.parquet", index=False)

print(f"daily.parquet:  {daily.shape[0]:,} filas · {daily.shape[1]} columnas")
print(f"videos.parquet: {videos.shape[0]:,} filas · {videos.shape[1]} columnas")

daily.parquet:  40,873 filas · 17 columnas
videos.parquet: 6,347 filas · 51 columnas


## 6. Diccionario del dataset por video

| Grupo | Columnas | Descripción |
|---|---|---|
| Identificación | `video_id`, `title`, `channel_title`, `category_id`, `category` | Atributos del primer día en tendencia |
| Texto | `tags`, `description` | Texto limpio; vacío si no había |
| Publicación | `publish_time`, `publish_hour`, `publish_dow`, `publish_day_name`, `publish_month`, `publish_is_weekend` | `publish_time` en UTC; el resto en hora del este de EE. UU. |
| Tendencia | `first_trending_date`, `last_trending_date`, `days_trending`, `days_to_trending` | Duración y velocidad de llegada |
| Marcas | `is_old_video`, `duration_incomplete`, `comments_disabled`, `ratings_disabled` | Casos a tratar aparte |
| Métricas | `views_*`, `likes_*`, `dislikes_*`, `comment_count_*` con sufijo `_first` o `_last` | Acumulados al primer y al último día |
| Logaritmos | `log_*` | `log(1 + x)` de cada métrica |
| Interacción | `like_rate`, `comment_rate`, `dislike_share`, `views_growth` | Calculadas al último día |
| Título | `title_length`, `title_words`, `title_upper_ratio`, `title_has_exclamation`, `title_has_question`, `title_has_digit` | Forma del título |
| Etiquetas y descripción | `n_tags`, `description_length`, `description_links` | Cantidad de etiquetas, longitud y enlaces |
| Canal | `channel_videos` | Videos del canal en tendencia durante el periodo |

## 7. Resumen

- De 40.949 filas se pasó a 40.873 filas diarias limpias y a **6.347 videos**.
- Se excluyeron 4 videos retirados; no se eliminó ningún otro. Los casos especiales quedan **marcados**, no borrados, para decidir en cada análisis si entran.
- El 16,5% de los videos tiene duración incompleta y el 1,6% son videos antiguos.
- El dataset queda listo para el análisis exploratorio (notebook 03).